# Secondary RQ — Regular public transit & communication apprehension

**Sample:** Prolific File A + File B stacked ∩ Qualtrics File C, matched on `Q0` / `Participant id`, with complete PRCA group + interpersonal ground truth.

**Secondary research question:** Do individuals who take public transportation **regularly** have communication-apprehension (CA) scores that differ statistically from the larger matched cohort? If so, by how much — and what does the score distribution look like?

**Primary exposure (Q26):** public transportation days in the last 3 months.  
**Regular riders** = `4-8 days a month` **or** `8 or more days a month` (weekly-or-more).

This notebook:

1. Loads / cleans the matched analytic sample  
2. Labels regular vs non-regular riders  
3. Describes CA distributions  
4. Runs Welch *t*-tests + Mann–Whitney + bootstrap CIs + effect sizes  
5. Checks sensitivity across alternate Q26 cutoffs  
6. Writes distributable artifacts under `outputs/transit_ca/`


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ca_personas.load import load_full_cohort
from ca_personas.paths import sibling_data_available
from ca_personas.transit_ca import (
    PRIMARY_REGULAR_LABELS,
    run_transit_ca_analysis,
    save_transit_ca_artifacts,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

assert sibling_data_available(), (
    "Place File A/B/C under ../sibling_data/ before running this notebook."
)
print("Regular-rider labels:", sorted(PRIMARY_REGULAR_LABELS))


## 1. Load matched analytic sample

Inner join of stacked Prolific waves with Qualtrics File C; keep respondents with complete PRCA items (`Q1–Q6`, `Q13–Q18`).


In [ ]:
participants, cleaning_report = load_full_cohort(join_how="inner")
print(json.dumps(cleaning_report, indent=2))
print("Analytic N:", len(participants))
participants[["participant_id", "Q26", "gt_group_ca", "gt_interpersonal_ca"]].head()


## 2. Run secondary-RQ analysis pipeline

Welch *t*-test (primary), Mann–Whitney (nonparametric sensitivity), Cohen's *d* / Hedges' *g*, and bootstrap 95% CIs for the mean difference (regular − not regular). Also reports regular mean vs overall cohort mean for the “how much vs the larger population” contrast.


In [ ]:
analysis = run_transit_ca_analysis(participants, n_boot=5000, random_state=42)
summary = analysis["summary"]
print(json.dumps(summary, indent=2))


## 3. Exposure frequencies (Q26)


In [ ]:
labeled = analysis["labeled"]
exposure = labeled.dropna(subset=["regular_transit"])
print(exposure["Q26"].value_counts(dropna=False))
print()
print(exposure["transit_group"].value_counts())
print(
    f"Regular share: {(exposure['regular_transit'] == True).mean():.1%} "
    f"({int((exposure['regular_transit'] == True).sum())}/{len(exposure)})"
)


## 4. Descriptive CA by transit group


In [ ]:
analysis["descriptives"]


In [ ]:
analysis["by_q26"]


## 5. Inferential tests — regular vs not-regular (and vs overall mean)


In [ ]:
comps = analysis["comparisons"].copy()
show_cols = [
    "score", "n_regular", "n_not_regular",
    "mean_regular", "mean_not_regular", "mean_overall",
    "diff_regular_minus_not_regular", "diff_regular_minus_overall",
    "pct_diff_vs_overall",
    "welch_t", "welch_p", "mannwhitney_p",
    "cohens_d", "hedges_g",
    "boot_ci_low", "boot_ci_high", "significant_at_05",
]
comps[show_cols]


In [ ]:
# Plain-language results card
for row in summary["primary_tests"]:
    score = "Group CA" if row["score"] == "gt_group_ca" else "Interpersonal CA"
    direction = "lower" if row["diff_regular_minus_overall"] < 0 else "higher"
    sig = "YES" if row["significant_at_05"] else "no"
    print(
        f"{score}: regular M={row['mean_regular']:.2f} vs overall M={row['mean_overall']:.2f} "
        f"(Δ={row['diff_regular_minus_overall']:+.2f} pts, {row['pct_diff_vs_overall']:+.1f}%); "
        f"vs non-regular Δ={row['diff_regular_minus_not_regular']:+.2f} "
        f"[95% boot CI {row['boot_ci_low']:+.2f}, {row['boot_ci_high']:+.2f}]; "
        f"Welch p={row['welch_p']:.4f}; Cohen d={row['cohens_d']:.3f}; significant@0.05={sig}"
    )
print()
print("Verdict:", summary["verdict"]["interpretation"])


## 6. Distributions for reporting


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
colors = {"regular": "#C5050C", "not_regular": "#555555"}
for ax, score, title in zip(
    axes,
    ["gt_group_ca", "gt_interpersonal_ca"],
    ["Group CA", "Interpersonal CA"],
):
    for group, color in colors.items():
        vals = labeled.loc[labeled["transit_group"] == group, score].dropna()
        ax.hist(vals, bins=range(6, 32), alpha=0.55, label=group, color=color, edgecolor="white")
        ax.axvline(vals.mean(), color=color, linestyle="--", linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel("PRCA subscale (6–30)")
axes[0].set_ylabel("Participants")
axes[0].legend(frameon=False, title="Transit group")
fig.suptitle("CA score distributions: regular vs not-regular public-transit riders")
fig.tight_layout()
OUT = ROOT / "outputs" / "transit_ca"
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "fig_ca_distributions_by_transit.png", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, score, title in zip(
    axes,
    ["gt_group_ca", "gt_interpersonal_ca"],
    ["Group CA", "Interpersonal CA"],
):
    data = [
        labeled.loc[labeled["transit_group"] == "regular", score].dropna(),
        labeled.loc[labeled["transit_group"] == "not_regular", score].dropna(),
        labeled[score].dropna(),
    ]
    bp = ax.boxplot(data, tick_labels=["Regular", "Not regular", "Overall"], patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#C5050C", "#888888", "#1a1a1a"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.35)
    ax.set_title(title)
    ax.set_ylabel("PRCA score")
fig.suptitle("CA score spread by transit group")
fig.tight_layout()
fig.savefig(OUT / "fig_ca_boxplots_by_transit.png", dpi=150)
plt.show()


## 7. Band prevalence (low / moderate / high)


In [ ]:
analysis["bands"]


## 8. Sensitivity — alternate definitions of “regular”


In [ ]:
sens = analysis["sensitivity"]
sens[[
    "cutoff", "score", "n_regular", "n_not_regular",
    "mean_regular", "mean_not_regular", "mean_overall",
    "diff_regular_minus_not_regular", "diff_regular_minus_overall",
    "welch_p", "cohens_d", "significant_at_05", "regular_labels",
]]


## 9. Write distributable artifacts

Tables + JSON results card land in `outputs/transit_ca/` (gitignored). Use these for the manuscript / slides.


In [ ]:
paths = save_transit_ca_artifacts(analysis, OUT)
# Also persist the cleaned analytic sample used here.
processed = ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)
participants.to_csv(processed / "participants_scored.csv", index=False)
(processed / "cleaning_report.json").write_text(json.dumps(cleaning_report, indent=2))
{k: str(v) for k, v in paths.items()}


## 10. Takeaway (auto-filled from results card)

The cell below prints the takeaway from `summary` / `transit_ca_summary.json` so this section never carries blank placeholders.


In [ ]:
def _cohen_size(d: float) -> str:
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"

sample = summary["sample"]
print("Regular definition:", sample["regular_definition"])
print(f"N regular / N comparison: {sample['n_regular']} / {sample['n_not_regular']}")
for row in summary["primary_tests"]:
    label = "Group CA" if row["score"] == "gt_group_ca" else "Interpersonal CA"
    print(
        f"{label}: mean difference vs overall = {row['diff_regular_minus_overall']:+.2f} pts; "
        f"Welch p = {row['welch_p']:.4g}; d = {row['cohens_d']:.3f} "
        f"({_cohen_size(row['cohens_d'])})."
    )
any_sig = summary["verdict"]["any_subscale_significant_at_05"]
ds = [abs(r["cohens_d"]) for r in summary["primary_tests"]]
mag = _cohen_size(max(ds)) if ds else "unknown"
print(
    "Conclusion: regular riders "
    + ("DO" if any_sig else "do NOT")
    + " differ significantly from the larger cohort under α = .05; "
    f"largest |d| magnitude is {mag}."
)
print("Interpretation:", summary["verdict"]["interpretation"])
